# 01 - IOI baseline

*Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2*

This notebook checks that GPT-2 small actually performs the Indirect Object Identification task on the evaluation dataset, and records the baseline measurements that later notebooks use for normalisation.

Model: GPT-2 small (Radford et al., 2019), accessed through TransformerLens (Nanda and Bloom, 2022). Task and reference circuit: Wang et al. (2023).

Run on Google Colab with an NVIDIA T4 GPU. Random seed fixed at 0.

In [1]:
!pip -q install transformer_lens==3.6.0

In [2]:
import torch
import random
import numpy as np
from transformer_lens import HookedTransformer

# no training happens here, so gradients are switched off
torch.set_grad_enabled(False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
print("GPT-2 small:", n_layers, "layers x", n_heads, "heads =", n_layers * n_heads, "attention heads")
print("device:", device)

/tmp/ipykernel_3036/976946140.py:10: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
GPT-2 small: 12 layers x 12 heads = 144 attention heads
device: cuda


## Building the dataset

Two templates are used rather than one, so the results do not depend on a single phrasing. Both are the same length in tokens. That matters because the sentences are processed in a batch and the metric is read from the last position of each one - if the lengths differed, shorter sentences would be padded and the metric would be read from the wrong place.

Only names that GPT-2 represents as a single token are used, because the metric compares the logits of individual answer tokens.

In [3]:
templates = [
    "When{A} and{B} went to the shop,{S} gave a drink to",
    "When{A} and{B} got to the office,{S} sent a letter to",
]

candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]

# keep only the names that are a single token
names = []
for name in candidate_names:
    n_tokens = model.to_tokens(name, prepend_bos=False).shape[1]
    if n_tokens == 1:
        names.append(name)

# check both templates come out the same length in tokens
template_lengths = []
for template in templates:
    example = template.format(A=" Mary", B=" John", S=" John")
    template_lengths.append(model.to_tokens(example).shape[1])

print("template lengths:", template_lengths)
assert template_lengths[0] == template_lengths[1], "templates must be the same length in tokens"

print("single-token names:", names)

template lengths: [15, 15]
single-token names: [' Mary', ' John', ' Tom', ' James', ' Anna', ' Kate', ' Mark', ' Paul', ' Alice', ' Sarah', ' David', ' Emma']


In [4]:
# build every ordered pair of two different names
all_pairs = []
for name_a in names:
    for name_b in names:
        if name_a != name_b:
            all_pairs.append((name_a, name_b))

# shuffle with a fixed seed and take 50 pairs
random.seed(0)
random.shuffle(all_pairs)
pairs = all_pairs[:50]

# each pair goes into both templates
prompts = []
io_tokens = []   # the correct answer token for each sentence
s_tokens = []    # the incorrect answer token for each sentence

for name_a, name_b in pairs:
    for template in templates:
        # name_b is the subject of the second clause, so name_a is the correct answer
        prompts.append(template.format(A=name_a, B=name_b, S=name_b))
        io_tokens.append(model.to_single_token(name_a))
        s_tokens.append(model.to_single_token(name_b))

tokens = model.to_tokens(prompts)
N = len(prompts)

print("name pairs:", len(pairs), "| templates:", len(templates), "| sentences:", N)
print("sequence length:", tokens.shape[1], "tokens")
print("example sentence:", repr(prompts[0]))

name pairs: 50 | templates: 2 | sentences: 100
sequence length: 15 tokens
example sentence: 'When Anna and Mark went to the shop, Mark gave a drink to'


## The behavioural metric

The logit difference is the logit of the correct name minus the logit of the incorrect name, taken at the final position and averaged over the sentences.

Accuracy is not used as the main measure because it is binary. Under ablation the model can become less confident well before its answer actually changes, and accuracy would show nothing until the behaviour collapsed.

In [5]:
def logit_difference_per_sentence(logits):
    """Return the logit difference for each sentence as a numpy array."""
    # logits has shape [sentence, position, vocabulary]
    # only the prediction at the final position is needed
    final_logits = logits[:, -1, :]

    values = []
    for i in range(N):
        correct_logit = final_logits[i, io_tokens[i]]
        wrong_logit = final_logits[i, s_tokens[i]]
        values.append((correct_logit - wrong_logit).item())

    return np.array(values)

In [6]:
logits = model(tokens)
per_sentence = logit_difference_per_sentence(logits)

n_correct = 0
for value in per_sentence:
    if value > 0:
        n_correct = n_correct + 1

print("sentences:            ", N)
print("mean logit difference:", round(float(per_sentence.mean()), 3))
print("standard deviation:   ", round(float(per_sentence.std()), 3))
print("accuracy:             ", round(100 * n_correct / N, 1), "%")

sentences:             100
mean logit difference: 3.262
standard deviation:    1.018
accuracy:              100.0 %


## A few example predictions

Worth looking at directly rather than trusting the averages.

In [7]:
for i in range(5):
    single_logits = model(model.to_tokens(prompts[i]))
    best_token = single_logits[0, -1].argmax()
    prediction = model.to_string(best_token)
    print(repr(prompts[i]))
    print("   predicted:", repr(prediction), "| logit difference:", round(float(per_sentence[i]), 2))
    print()

'When Anna and Mark went to the shop, Mark gave a drink to'
   predicted: ' Anna' | logit difference: 3.51

'When Anna and Mark got to the office, Mark sent a letter to'
   predicted: ' Anna' | logit difference: 2.95

'When Mary and Anna went to the shop, Anna gave a drink to'
   predicted: ' Mary' | logit difference: 2.1

'When Mary and Anna got to the office, Anna sent a letter to'
   predicted: ' Mary' | logit difference: 2.72

'When Emma and Anna went to the shop, Anna gave a drink to'
   predicted: ' Emma' | logit difference: 2.4



## Result

GPT-2 small performs IOI reliably on this dataset. The mean logit difference is clearly positive and accuracy sits at its maximum.

Accuracy being at ceiling is itself the reason for using logit difference throughout the rest of the study - a binary measure has nowhere left to move, so it would register nothing until the behaviour failed completely.

These baseline values are used in notebooks 02 to 04 to normalise the importance scores and to interpret the faithfulness curves.

### References

- Nanda, N. and Bloom, J. (2022) *TransformerLens*.
- Radford, A. et al. (2019) *Language Models are Unsupervised Multitask Learners*.
- Wang, K. et al. (2023) *Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small*. ICLR.